In [1]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
#import rasterio
#from rasterio.mask import mask

In [46]:
project_root = os.path.dirname(os.path.dirname("test.ipynb"))
data_dir = os.path.join(project_root, "data")
raster_dir = os.path.join(data_dir, "rasters")
transient_dir = os.path.join(project_root, "transients")
output_dir = os.path.join(project_root, "outputs")

energy_raster_path = os.path.join(raster_dir, "GHS_BUILT_S_timeseries_points.gpkg")


shapefile_path = os.path.join(data_dir, "ne_10m_admin_0_countries.shp")  # may need other files rather than just shp?
raster_path = os.path.join(data_dir, "gpw_v4_population_density_rev11_2020_30_min.tif")
country_energy_path = os.path.join(data_dir, "Country Energy Data.xlsx")

In [47]:
energy_timeseries = gpd.read_file(energy_raster_path)

year = 2025
place_ocean = True
all_leds_gdf = gpd.GeoDataFrame()

In [94]:
led_data = pd.read_csv("data/API/global_energy_consumption.csv")
led_data.pivot(index="Entity", columns="Year", values="primary_energy_consumption__twh")

Year,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
Entity,,,,,,,,,,,,,,,,,,,,,
Afghanistan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,30.953520,28.075348,36.517418,46.492510,47.111530,46.911743,47.569305,45.106330,41.059685,NaN
Africa,715.2595,749.1478,757.0225,799.8021,821.88776,892.29584,951.0875,1006.129,1091.3364,1146.7285,...,5163.164000,5280.843000,5444.701000,5530.572800,5681.832500,5412.605000,5870.373500,5954.654300,6050.292000,6139.628
Africa (EI),715.2595,749.1478,757.0225,799.8021,821.88776,892.29584,951.0875,1006.129,1091.3364,1146.7285,...,5163.164000,5280.843000,5444.701000,5530.572800,5681.832500,5412.605000,5870.373500,5954.654300,6050.292000,6139.628
Africa (EIA),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5425.935000,5464.726000,5698.093300,5717.813500,5916.964400,5551.270000,5772.613300,5965.856400,6051.291000,NaN
Albania,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,24.037636,26.761011,29.290280,27.993164,24.897661,21.116934,25.489720,25.864765,22.583450,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
World,43360.3100,45732.4500,47410.5550,50246.7460,53700.40200,57177.61300,59438.5620,62703.992,66337.6500,66696.5100,...,152357.620000,154221.480000,157625.310000,161806.120000,163694.610000,157993.890000,166043.500000,169061.530000,172238.780000,176737.100
Yemen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,52.187027,41.672176,38.380810,37.804485,38.115673,33.195854,32.063576,34.169525,34.487343,NaN
Yugoslavia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [78]:
led_data = pd.read_csv("data/API/global_energy_consumption.csv")
led_data = led_data.pivot(index="Entity", columns="Year", values="primary_energy_consumption__twh").reset_index()

missing = max(led_data.isna().sum())
idx = (np.abs(led_data.columns.values - year)).argmin()
energy_year = led_data.columns.values[idx]
while missing > 10:
    missing = led_data.isna().sum().loc[energy_year]
    if missing > 10:
        energy_year -= 1

TypeError: unsupported operand type(s) for -: 'str' and 'int'

In [45]:
for index, row in led_data.iterrows():

    country_name = row['Entity']
    num_leds = int(row['Round'])
    leds_placed = 0

    values_array = energy_timeseries[energy_timeseries['country'] == country_name]
    values_array = values_array[["point_index", f"{year}", "geometry"]].sort_values(f"{year}", ascending=False)

    available_cells = len(values_array)
    missing_leds = num_leds - available_cells
    
    if available_cells == 0:
        print(f"Could not find {country_name} in raster data, skipping...")

    else:

        for leds in range(0,num_leds-leds_placed): # Place LEDs on the land-space
            
            if leds_placed < available_cells: 
                
                all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_array["geometry"].iloc[leds]],
                                                                        'Country': [country_name],
                                                                        'Raster_Density': [values_array[f"{year}"].iloc[leds]]
                                                                        }, geometry='geometry')], ignore_index=True)
                leds_placed += 1


            else:

                if place_ocean == True:
                    
                    if leds_placed >= num_leds:
                        break

                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {available_cells} cells are available. Attempting to place {missing_leds} LEDs in surrounding area.")
                    values_sorted = energy_timeseries.sort_values('point_index').reset_index(drop=True)
                    surround = 1
                    
                    while leds_placed < num_leds:

                        surround_indices = ([x - (720*surround) for x in values_array.point_index] + 
                                            [x + (720*surround) for x in values_array.point_index] +
                                            [x - surround for x in values_array.point_index] + 
                                            [x + surround for x in values_array.point_index]) 
                        surround_indices = np.unique(list(filter(lambda x: x >= 0, surround_indices)))
                        values_filtered = energy_timeseries[energy_timeseries["point_index"].isin(surround_indices)].query("country.isnull()")

                        for leds in range(len(values_filtered)): # Place remaining LEDs on the land-space
                            all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_filtered["geometry"].iloc[leds]],
                                                                                    'Country': [country_name],
                                                                                    'Raster_Density': [values_filtered[f"{year}"].iloc[leds]]
                                                                                    }, geometry='geometry')], ignore_index=True)
                            leds_placed += 1
                            if leds_placed >= num_leds:
                                break
                        surround += 1
                
                else: 
                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {leds_placed} were placed due to not having enough space.")
                    break

KeyError: 'Entity'

In [44]:
all_leds_gdf["geometry"] 

0       POINT (113.24875 22.84958)
1       POINT (121.24875 31.34958)
2       POINT (116.24875 39.84958)
3       POINT (120.24875 31.84958)
4       POINT (113.74875 22.84958)
                   ...            
6887     POINT (74.74875 42.84958)
6888     POINT (24.74875 59.34958)
6889    POINT (-87.75125 15.34958)
6890     POINT (44.24875 40.34958)
6891     POINT (28.74875 46.84958)
Name: geometry, Length: 6892, dtype: geometry

In [62]:
all_leds_gdf[all_leds_gdf['Country'] == "South Korea"]["geometry"]

2234    POINT (126.74875 37.34958)
2235    POINT (127.24875 37.34958)
2236    POINT (127.24875 36.84958)
2237    POINT (128.74875 35.34958)
2238    POINT (129.24875 35.34958)
                   ...            
2300    POINT (127.24875 33.34958)
2301    POINT (126.24875 32.84958)
2302    POINT (126.74875 32.84958)
2303    POINT (128.24875 39.34958)
2304    POINT (128.24875 38.84958)
Name: geometry, Length: 71, dtype: geometry

In [63]:
all_leds_gdf.to_file("dataframe.gpkg", driver="GPKG")

c:\ProgramData\miniforge3\envs\science_gen\Lib\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
